> **SOLUTIONS notebook.** Answers are filled into the `#answer` cells below; Try the exercises yourself first.

This file is part of the CRISPRsummerschool 2026 exercises

Copyright (c) 2023-26 Christian Anthon & 2026 Gül Sude Demircan

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, version 3.

# Warming up exercise
Below you will find the first exercise, in which you will be introduced a small CRISPR on-target model in PyTorch and use it to train a small ontarget efficiency model on real data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RTH-tools/CRISPRsummerschool/blob/main/2026/CRISPR/exercise/crispr_2026_crispr_exercise1.ipynb)


## basic code definitions
Enter the cell below and press play or Ctrl+Enter in the block below to execute. You should see the message "Definitions executed" printed after execution.

In [1]:
#!/usr/bin/env python3
# CRISPRsummerschool 2026 -- PyTorch version
import os
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


eLENGTH30 = 30   # sequence length: the target region is a 30-mer, so 30 input positions
eDEPTH    = 4    # one-hot channels per position: the 4 nucleotides A, C, G, T

# Function to convert DNA sequence to an index encoding (A,C,G,T/U -> 0,1,2,3)
def onehot(x):
    z = list()
    for y in list(x):
        if y in "Aa":
            z.append(0)
        elif y in "Cc":
            z.append(1)
        elif y in "Gg":
            z.append(2)
        elif y in "TtUu":
            z.append(3)
        else:
            print("Non-ATGCU character in", x)
            raise Exception
    return z

# Function to set the data into the appropriate format
def set_data(DX, s):
    if s is None:
        return
    for j, x in enumerate(onehot(s)):
        DX[j][x] = 1

# Preprocessing function for the sequence data
def preprocess_seq(data):
    DATA_X30 = np.zeros((len(data), eLENGTH30, eDEPTH), dtype=np.float32)  # onehot
    DATA_G = np.zeros((len(data), 1), dtype=np.float32)  # deltaGb
    DATA_Y = np.zeros((len(data)), dtype=np.float32)  # efficiency

    for l, d in enumerate(data):
        set_data(DATA_X30[l], d[1])
        DATA_G[l] = -d[2]
        DATA_Y[l] = d[3]
    return (DATA_X30, DATA_G, DATA_Y)


# Convert numpy arrays to torch tensors on `device`.
# NOTE: the one-hot stays (N, 30, 4) here; the model permutes it to
#       (N, 4, 30) internally, because PyTorch Conv1d expects the layout
#       (batch, channels, length).
def to_tensors(x30, g, y):
    return (torch.from_numpy(x30).to(device),
            torch.from_numpy(g).to(device),
            torch.from_numpy(y).to(device))

def evaluate(model, data):
    """Return (mse, mae) on a (Xc, Xg, y) split. Runs in eval mode (dropout OFF)."""
    Xc, Xg, y = data
    model.eval()
    with torch.no_grad():
        pred = model(Xc, Xg).squeeze(-1)          # (N, 1) -> (N,)
        mse = torch.mean((pred - y) ** 2).item()
        mae = torch.mean(torch.abs(pred - y)).item()
    return mse, mae

def train(model, train_data, val_data, epochs=200, batch_size=64, lr=1e-3,
          patience=25, min_delta=0.1, verbose=True):
    """Mini-batch training with early stopping and restore-best-weights."""
    Xc, Xg, y = train_data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait, history = float("inf"), None, 0, []
    n = Xc.shape[0]
    for epoch in range(epochs):
        model.train()                             # dropout ON
        perm = torch.randperm(n, device=Xc.device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            pred = model(Xc[idx], Xg[idx]).squeeze(-1)   # (B,1) -> (B,): MUST squeeze
            loss = loss_fn(pred, y[idx])
            loss.backward()
            optimizer.step()
        val_mse, val_mae = evaluate(model, val_data)
        history.append(val_mse)
        if val_mse < best_val - min_delta:        # "minimum improvement" rule
            best_val, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if verbose:
            print("epoch %3d  val_mse=%8.3f  val_mae=%6.3f  best=%8.3f  wait=%d"
                  % (epoch, val_mse, val_mae, best_val, wait))
        if wait >= patience:
            print("Early stopping at epoch %d (best val_mse=%.3f)" % (epoch, best_val))
            break
    if best_state is not None:
        model.load_state_dict(best_state)         # restore_best_weights=True
    return history


# ---- Robust data loader: identical behaviour in Colab and local Jupyter ----
# A file already in this folder is used as-is; a missing file is downloaded and
# its contents are validated. Pure Python (no shell), so it behaves the same in
# Colab, local Jupyter, Windows/Mac/Linux.
import urllib.request

DATA_SOURCES = {
    "training_data.csv": [
        "https://rth.dk/internal/index.php/s/S4jQMaER6nYAJGe/download",
    ],
    "validation_data.csv": [
        "https://rth.dk/internal/index.php/s/oHspJCgniRMog6r/download",
    ],
}

def _is_valid_csv(path):
    """A real data file starts with the known header, not an HTML error page."""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            first = fh.readline()
        return ("target" in first) and ("deltaGb" in first)
    except OSError:
        return False

def fetch_data(fname, dest_dir="."):
    """Return the path to a valid `fname`, downloading it only if needed.
    urllib raises on HTTP errors (unlike a bare `curl -o`) and we re-check the
    content, so a bad/expired URL fails loudly instead of silently writing an
    HTML page into a .csv."""
    dest = os.path.join(dest_dir, fname)
    if _is_valid_csv(dest):
        print("using existing", dest)
        return dest
    problems = []
    for url in DATA_SOURCES[fname]:
        try:
            print("downloading %s from %s ..." % (fname, url.split("/")[2]))
            urllib.request.urlretrieve(url, dest)
        except Exception as e:
            problems.append("%s -> %s" % (url, e))
            continue
        if _is_valid_csv(dest):
            return dest
        problems.append("%s -> downloaded file is not a valid CSV (an error page?)" % url)
    if os.path.exists(dest):
        os.remove(dest)                       # never leave a corrupt .csv behind
    raise RuntimeError(
        "Could not obtain %s. Upload it into this folder manually, or fix the "
        "URLs in DATA_SOURCES above.\nTried:\n  %s" % (fname, "\n  ".join(problems)))

fetch_data("training_data.csv")
fetch_data("validation_data.csv")

print('\n\nDefinitions executed')


Using device: cpu
downloading training_data.csv from rth.dk ...
downloading validation_data.csv from rth.dk ...


Definitions executed


## Exercise 1.1
The sequence of the ontarget of an example gRNA (ACTGAAAAAACCCCCTTTTT), needs to be onehot encoded. An example ontarget of ACTGAAAAAACCCCCTTTTT is TTTTACTGAAAAAACCCCCTTTTTGGGAAA, which includes a four nucleotide prefix, the ontarget to the gRNA, the PAM sequnce and a three nucleotide suffix.

**Task 1.** Split the 30-mer `TTTTACTGAAAAAACCCCCTTTTTGGGAAA` into its four parts. For each part, give the subsequence and its 0-based slice range (`s[0:4]` style): prefix (4 nt), spacer / on-target (20 nt), PAM (3 nt), suffix (3 nt).

**Task 2.** Call `onehot("ACTGAAAAAACCCCCTTTTT")` (the function defined in the cell above) and print the result. State how many numbers it returns and which values they can take.

**Task 3.** A true one-hot encoding gives every base its own length-4 vector - `A = [1,0,0,0]`, `C = [0,1,0,0]`, `G = [0,0,1,0]`, `T = [0,0,0,1]` - so a 20-nt spacer becomes a `(20, 4)` array of 0s and 1s. Compare that with your Task 2 output, then answer:

1. Is the Task 2 output a one-hot encoding? If not, what kind of encoding is it?
2. Name one concrete way this encoding could mislead a model that feeds those numbers straight into a network.

In [2]:
# --- Exercise 1.1 ---

# Task 1: split the 30-mer (checked with slices, so there is nothing to trust)
s = "TTTTACTGAAAAAACCCCCTTTTTGGGAAA"
print("len          :", len(s))          # 30 = 4 + 20 + 3 + 3
print("prefix s[0:4]:", s[0:4])          # TTTT
print("spacer s[4:24]:", s[4:24])        # ACTGAAAAAACCCCCTTTTT   <- the on-target / gRNA
print("PAM  s[24:27]:", s[24:27])        # GGG                    <- the NGG, here N = G
print("suffix s[27:30]:", s[27:30])      # AAA

# Task 2: encode the spacer
code = onehot("ACTGAAAAAACCCCCTTTTT")
print("code   :", code)
print("n values:", len(code), " range:", min(code), "-", max(code))
# -> [0, 1, 3, 2, 0,0,0,0,0,0, 1,1,1,1,1, 3,3,3,3,3]
#    20 numbers (one per base), each in 0-3.

# Task 3:
# 1. NO, this is not a one-hot encoding. It is a LABEL (ordinal / integer) encoding:
#    one integer per base, A=0 C=1 G=2 T=3. A one-hot would be a (20, 4) array of
#    0s and 1s - 80 numbers, not 20 - with exactly one 1 per position.
# 2. How it misleads: integers carry order and distance that the bases do not have.
#    This code asserts A < C < G < T, and that G (2) is closer to C (1) than to A (0).
#    A linear layer computes w * x, so doubling the input doubles the contribution:
#    "T" would push a neuron three times as hard as "C", purely because of the label
#    we chose. The four bases are unordered categories, so each needs its own
#    independent input channel - which is what one-hot gives.

len          : 30
prefix s[0:4]: TTTT
spacer s[4:24]: ACTGAAAAAACCCCCTTTTT
PAM  s[24:27]: GGG
suffix s[27:30]: AAA
code   : [0, 1, 3, 2, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3]
n values: 20  range: 0 - 3



## Exercise 1.2
Excecute the code below to load the data into the notebook.

In [3]:
# x30 - onehot encoded 30mer
# g   - deltaGb
# y   - the efficiency value (~0 - ~100)

PATH = './'
d = pd.read_csv(PATH + 'training_data.csv').values.tolist()
(x30, g, y) = preprocess_seq(d)

# Validation data
dv = pd.read_csv(PATH + 'validation_data.csv').values.tolist()
(x30v, gv, yv) = preprocess_seq(dv)

# to torch tensors on `device`
x30_t, g_t, y_t = to_tensors(x30, g, y)
x30v_t, gv_t, yv_t = to_tensors(x30v, gv, yv)
print("train tensors:\n", x30_t.shape, "\n", g_t.shape, "\n", y_t.shape)

train tensors:
 torch.Size([15935, 30, 4]) 
 torch.Size([15935, 1]) 
 torch.Size([15935])


### Reading the printed shapes above

```
train tensors:
torch.Size([15935, 30, 4])
torch.Size([15935, 1])
torch.Size([15935])
```

All three are `float32` on `device`. **15935 = number of training guides**, and it is the first axis of all three: index `i` means the same guide in each.

| tensor | shape | axes |
|---|---|---|
| `x30_t` | `(15935, 30, 4)` | guide, position in the 30-mer, nucleotide channel `A,C,G,T` |
| `g_t` | `(15935, 1)` | guide, the energy feature (`-deltaGb`) |
| `y_t` | `(15935,)` | guide (measured efficiency) |


### Exercise 1.2.1

Print the first row of the raw training data (`d[0]`) and the first entry of each processed array (`x30[0]`, `g[0]`, `y[0]`).

1. Which field of `d[0]` did each of `x30[0]`, `g[0]`, `y[0]` come from? Do the values match?
2. Is `x30[0]` what a one-hot encoding should look like?

In [4]:
# --- Exercise 1.2.1 ---
print("raw d[0]     :", d[0])            # [idx, 30mer, deltaGb, eff, none]
print("x30[0] shape :", x30[0].shape)    # (30, 4)  one-hot
print("x30[0]:\n", x30[0])
print("g[0]  (-dGb) :", g[0])            # energy feature (positive)
print("y[0]  (eff)  :", y[0])
# Each row of x30[0] has exactly one 1 (the base at that position). g[0] is
# -deltaGb (column 2); y[0] is the efficiency (column 3). This is the proper
# one-hot we expected -- unlike the raw index list from Exercise 1.1.


raw d[0]     : [0, 'ATGTACACCCAAGGGTCCAGGATCTGGTTC', -29.0786235622, 35.3792174653667, nan]
x30[0] shape : (30, 4)
x30[0]:
 [[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]
g[0]  (-dGb) : [29.078623]
y[0]  (eff)  : 35.37922


### Exercise 1.2.2 Model definition
In the code below, a simplified version of the CRISPRon ontarget model is defined. Review the code without diving into the details. Then execute it to load the model.

In [5]:
DROPOUT_DENSE = 0.3   # dropout rate: fraction of units switched off after each dense layer (training only)
CONV_1_SIZE   = 3     # convolution kernel width, in nucleotides (a 3-base window)
N_CONV_1      = 40    # number of convolution filters (motif detectors)
N_DENSE       = 40    # width of the first dense layers (collect, dense1)
N_OUT         = 40    # width of the last hidden layers (dense2, dense3), before the 1-unit output

class SimpleCRISPRon(nn.Module):
    """A simplified version of the CRISPRon on-target model (PyTorch).

    One 1D convolution over the one-hot sequence, followed by fully connected
    (dense) layers. The binding energy dGb is concatenated in *after* the first
    dense layer.
    """
    def __init__(self, n_conv=N_CONV_1, kernel=CONV_1_SIZE, n_dense=N_DENSE,
                 n_out=N_OUT, dropout=DROPOUT_DENSE, seq_len=eLENGTH30, depth=eDEPTH):
        super().__init__()
        self.conv = nn.Conv1d(depth, n_conv, kernel)      # (B,4,30) -> (B,n_conv,28)
        conv_out_len = seq_len - kernel + 1               # 30 - 3 + 1 = 28
        flat = n_conv * conv_out_len                      # flattened conv features
        self.collect = nn.Linear(flat, n_dense)           # "dense_0"
        self.dense1 = nn.Linear(n_dense + 1, n_dense)     # "dense_1"  (+1 = raw dGb)
        self.dense2 = nn.Linear(n_dense, n_out)           # "dense_2"
        self.dense3 = nn.Linear(n_out, n_out)             # "dense_on_off"
        self.out = nn.Linear(n_out, 1)                    # output (linear, no activation)
        self.drop = nn.Dropout(dropout)
        self.apply(self._xavier_uniform_init)

    def features(self, x, g):
        # Conv1d wants (batch, channels, length); our one-hot is
        # (batch, length=30, channels=4), so swap the last two axes.
        x = x.permute(0, 2, 1)                            # (B,30,4) -> (B,4,30)
        z = torch.relu(self.conv(x))                      # convolution + ReLU
        z = torch.flatten(z, start_dim=1)                 # (B, n_conv*28)
        z = self.drop(torch.relu(self.collect(z)))        # dense_0 + ReLU + dropout
        z = torch.cat([g, z], dim=1)                      # concat raw dGb
        z = self.drop(torch.relu(self.dense1(z)))         # dense_1
        z = self.drop(torch.relu(self.dense2(z)))         # dense_2
        z = self.drop(torch.relu(self.dense3(z)))         # dense_on_off
        return z

    def forward(self, x, g):
        return self.out(self.features(x, g))              # (B, 1)

    @staticmethod
    def _xavier_uniform_init(m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

model = SimpleCRISPRon().to(device)
print(model)
print('Model defined')


SimpleCRISPRon(
  (conv): Conv1d(4, 40, kernel_size=(3,), stride=(1,))
  (collect): Linear(in_features=1120, out_features=40, bias=True)
  (dense1): Linear(in_features=41, out_features=40, bias=True)
  (dense2): Linear(in_features=40, out_features=40, bias=True)
  (dense3): Linear(in_features=40, out_features=40, bias=True)
  (out): Linear(in_features=40, out_features=1, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
)
Model defined


### Exercise 1.2.3

`print(model)` lists the layers, but not the order they run in. For that, read the `features` method in the cell above.

1. The model takes two inputs. Name them, and say which layer each one enters.
2. Which layer produces the prediction, and what is its output shape?
3. Sort the layers into the convolutional part and the fully connected (MLP) part.
4. Where does ΔGb join the sequence path, and what effect does that have on the layer receiving it?

Useful:

```python
for name, p in model.named_parameters():
    print(name, tuple(p.shape))     # weight shapes, layer by layer
```

In [6]:
# --- Exercise 1.2.3 ---
print(model)

# Trace the shapes through a dummy forward pass (batch of 8, dropout off):
model.eval()
with torch.no_grad():
    xb, gb = x30_t[:8], g_t[:8]
    print("input seq     :", tuple(xb.shape), "  input dGb:", tuple(gb.shape))
    z = xb.permute(0, 2, 1);          print("after permute :", tuple(z.shape))
    z = torch.relu(model.conv(z));    print("after conv    :", tuple(z.shape))
    z = torch.flatten(z, 1);          print("after flatten :", tuple(z.shape))
    z = torch.relu(model.collect(z)); print("after collect :", tuple(z.shape))
    z = torch.cat([gb, z], dim=1);    print("after cat dGb :", tuple(z.shape))   # <- dGb joins here
    z = torch.relu(model.dense1(z));  print("after dense1  :", tuple(z.shape))
    z = torch.relu(model.dense2(z));  print("after dense2  :", tuple(z.shape))
    z = torch.relu(model.dense3(z));  print("after dense3  :", tuple(z.shape))
    z = model.out(z);                 print("after out     :", tuple(z.shape))
print("trainable params:", sum(p.numel() for p in model.parameters()))

# 1. Two inputs: the one-hot 30-mer (8,30,4) -> enters `conv`; the scalar dGb (8,1)
#    -> does NOT touch the conv, it enters `dense1` via torch.cat.
# 2. Output: `out`, a single linear neuron (no activation) -> shape (8,1) = one
#    predicted efficiency per guide.
# 3. Convolutional part: `conv` (the sequence feature extractor).
#    MLP part: collect -> dense1 -> dense2 -> dense3 -> out.
# 4. dGb joins AFTER `collect`, at torch.cat([g, z], dim=1): 40 sequence features
#    + 1 energy value = 41, which is why dense1 is Linear(41, 40). Note the cat
#    order [g, z] puts dGb in column 0 of dense1's input.

SimpleCRISPRon(
  (conv): Conv1d(4, 40, kernel_size=(3,), stride=(1,))
  (collect): Linear(in_features=1120, out_features=40, bias=True)
  (dense1): Linear(in_features=41, out_features=40, bias=True)
  (dense2): Linear(in_features=40, out_features=40, bias=True)
  (dense3): Linear(in_features=40, out_features=40, bias=True)
  (out): Linear(in_features=40, out_features=1, bias=True)
  (drop): Dropout(p=0.3, inplace=False)
)
input seq     : (8, 30, 4)   input dGb: (8, 1)
after permute : (8, 4, 30)
after conv    : (8, 40, 28)
after flatten : (8, 1120)
after collect : (8, 40)
after cat dGb : (8, 41)
after dense1  : (8, 40)
after dense2  : (8, 40)
after dense3  : (8, 40)
after out     : (8, 1)
trainable params: 50361


### Exercise 1.2.4 Model training
Below you will find code for training the simplified CRISPRon model on the provided training data, using the validation data for model evaluation during training. Familiarize yourself with the code and parameters.

What is the difference between BATCH_SIZE and epochs?

Execute the model **training**

In [7]:
# --- Exercise 1.2.4: what is the difference between BATCH_SIZE and epochs? ---
#   BATCH_SIZE = how many training examples are used to compute ONE gradient
#     update; the weights are updated once per mini-batch.
#   EPOCHS     = how many full passes over the ENTIRE training set. Each epoch
#     does (n_samples / BATCH_SIZE) weight updates.
#   Smaller batches -> more, noisier updates per epoch; larger batches -> fewer,
#   smoother updates. Epochs control how many times the model sees all the data.

print("training...")

LEARN = 1e-3       # learning rate for Adam
EPOCHS = 200       # maximum number of epochs
BATCH_SIZE = 64    # batch size for the training

# (to restart training from scratch, re-run the model-definition cell in 1.2.2)
history = train(model, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
                patience=25, min_delta=0.1)

print("done")
val_mse, val_mae = evaluate(model, (x30v_t, gv_t, yv_t))
print("validation  mse=%.3f  mae=%.3f" % (val_mse, val_mae))


training...
epoch   0  val_mse= 309.452  val_mae=14.038  best= 309.452  wait=0
epoch   1  val_mse= 273.430  val_mae=13.298  best= 273.430  wait=0
epoch   2  val_mse= 270.727  val_mae=13.335  best= 270.727  wait=0
epoch   3  val_mse= 276.054  val_mae=13.555  best= 270.727  wait=1
epoch   4  val_mse= 249.743  val_mae=12.775  best= 249.743  wait=0
epoch   5  val_mse= 235.293  val_mae=12.453  best= 235.293  wait=0
epoch   6  val_mse= 245.633  val_mae=12.746  best= 235.293  wait=1
epoch   7  val_mse= 250.716  val_mae=12.900  best= 235.293  wait=2
epoch   8  val_mse= 254.722  val_mae=13.018  best= 235.293  wait=3
epoch   9  val_mse= 272.459  val_mae=13.565  best= 235.293  wait=4
epoch  10  val_mse= 209.890  val_mae=11.655  best= 209.890  wait=0
epoch  11  val_mse= 248.325  val_mae=12.904  best= 209.890  wait=1
epoch  12  val_mse= 256.143  val_mae=13.076  best= 209.890  wait=2
epoch  13  val_mse= 245.857  val_mae=12.739  best= 209.890  wait=3
epoch  14  val_mse= 210.194  val_mae=11.646  best=

### Exercise 1.2.4.1
How many epochs did the code use before it stopped?

When did the training reach the optimimal model?

Does the code output the exact same performance if you run it twice? Why / Why not?

In [8]:
# --- Exercise 1.2.4.1 ---
print("epochs actually run :", len(history))
print("lowest val MSE seen :", min(history), "at epoch", int(np.argmin(history)))
print("restored model      :", evaluate(model, (x30v_t, gv_t, yv_t)))

# How many epochs, and when was the optimum reached?
#   Early stopping kept training for `patience` (25) more epochs after the last
#   accepted improvement, then restore_best_weights rolled the weights back to it.
#   Careful: train() only ACCEPTS an improvement when val_mse < best - min_delta
#   (0.1). So the restored weights are the last accepted improvement, which is not
#   necessarily min(history) - that is why the two prints above can differ slightly.
#
# Does it give the exact same performance if you run it twice? It depends on what
# you re-run:
#   * Fresh kernel, all cells top to bottom -> YES, reproducible. The definitions
#     cell calls torch.manual_seed(0), so the Xavier init, the dropout masks and
#     the batch shuffling all replay identically. (Bitwise on the same machine and
#     torch build; on a GPU the last digits can still differ.)
#   * Re-running only this training cell in the same kernel -> NO, for two
#     independent reasons:
#       1. the seed is set once, and every random draw advances the global RNG, so
#          the second run starts from a different point in the stream;
#       2. it keeps training the ALREADY-trained model instead of starting over -
#          re-run the model-definition cell (1.2.2) first to start from scratch.


epochs actually run : 92
lowest val MSE seen : 164.2148895263672 at epoch 66
restored model      : (164.2148895263672, 10.066642761230469)


### Exercise 1.2.4.2
Repeat the model initialization in (1.2.2) and the model training (1.2.4) 3-5 times and record the performance on the validation data each time to see how much the result varies.

The model in this notebook is a deliberately simplified CRISPRon: one small convolution and narrow dense layers, so it trains in minutes. The published CRISPRon (Xiang et al., 2021) uses the same principles with more and larger layers, and reaches **MSE 141.3, MAE 9.1** on this same validation data. Compare the mean squared error and mean absolute error you got on the validation data with the errors obtained for the original published model.

- How far is your best run from those numbers?
- How much do your own runs differ from each other?



In [9]:
# --- Exercise 1.2.4.2 ---
results = []
for run in range(3):
    torch.manual_seed(run)
    m = SimpleCRISPRon().to(device)
    train(m, (x30_t, g_t, y_t), (x30v_t, gv_t, yv_t),
          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARN,
          patience=25, min_delta=0.1, verbose=False)
    mse, mae = evaluate(m, (x30v_t, gv_t, yv_t))
    results.append((mse, mae)); print(f"run {run}: mse={mse:.1f}  mae={mae:.2f}")
print("best simplified model:", min(results))
# The FULL CRISPRon reaches mae 9.1 / mse 141.3 on this data. This simplified
# model (one small conv, narrow dense layers) is a bit worse -- typically
# mse ~160-180, mae ~10-11 -- because it has far less capacity.


Early stopping at epoch 91 (best val_mse=164.215)
run 0: mse=164.2  mae=10.07
Early stopping at epoch 71 (best val_mse=164.172)
run 1: mse=164.2  mae=10.09
Early stopping at epoch 63 (best val_mse=164.290)
run 2: mse=164.3  mae=10.13
best simplified model: (164.17210388183594, 10.085619926452637)


### Exercise 1.2.5 (if time allows)
Play with the model and parameters to get a better performance.

Can you beat the original model performance?

What would be the proper way to **test** that?

In [10]:
# --- Exercise 1.2.5 (open-ended) ---
# Ideas to try: more filters (N_CONV_1), wider/deeper dense layers, a second
# convolution, different learning rate or batch size, longer patience.
#
# Can you beat the original model (mse 141.3)?  Hard with this single-conv design --
# the original CRISPRon uses several parallel convolutions (kernels 3/5/7) and wider
# layers.
#
# The PROPER way to claim you beat it:
#  - Do NOT tune hyper-parameters on the validation set and then report
#    validation numbers -- that leaks information about that split.
#  - Keep a separate, untouched TEST set (or use cross-validation): tune on
#    validation, then report ONCE on the held-out test set. Average over several
#    random seeds so you can tell a real gain from run-to-run noise.


## Reference

These exercises use a **simplified** version of the CRISPRon on-target efficiency model:

- Xiang, X., Corsi, G. I., Anthon, C., *et al.* (2021). Enhancing CRISPR-Cas9 gRNA efficiency prediction by data integration and deep learning. *Nature Communications*, **12**, 3238. https://doi.org/10.1038/s41467-021-23576-0
